In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
import shutil
import torch
from torchvision import datasets
from PIL import Image, ImageEnhance
import numpy as np
import random

# CONFIGURATION

KAGGLE_WORKING = '/kaggle/working'
REPO_DIR = f'{KAGGLE_WORKING}/RethinkFL'
DATA_ROOT = f'{REPO_DIR}/datasets'

# Setup Environment

os.chdir(KAGGLE_WORKING)

# Remove old repo if exists
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

os.system("pip install setproctitle -q")
print("Environment ready")

# Clone the Fork

os.system("git clone https://github.com/23b2232/RethinkFL.git")

os.chdir(REPO_DIR)
print(f"   Working directory: {os.getcwd()}")

# Verify the Changes

# Check if sample_clients method exists
with open('models/utils/federated_model.py', 'r') as f:
    if 'def sample_clients' in f.read():
        print("sample_clients() method found in FederatedModel")
    else:
        print("sample_clients() method NOT found")

# Check if models use it
for model_file in ['models/moon.py', 'models/fpl.py', 'models/fedavg.py', 'models/feddyn.py']:
    with open(model_file, 'r') as f:
        if 'self.sample_clients(total_clients)' in f.read():
            print(f"   {model_file} uses sample_clients()")
        else:
            print(f"   {model_file} NOT updated!")

# Check if main.py calls add_sampling_args
with open('main.py', 'r') as f:
    if 'add_sampling_args(parser)' in f.read():
        print("main.py calls add_sampling_args()")
    else:
        print("main.py missing add_sampling_args()")

# Setup Datasets

os.makedirs(DATA_ROOT, exist_ok=True)
domains = ['mnist', 'svhn', 'usps', 'syn']

def organize_dataset(dataset, domain, split, max_samples=None):
    for label in range(10):
        os.makedirs(f'{DATA_ROOT}/{domain}/{split}/{label}', exist_ok=True)
    
    num_samples = len(dataset) if max_samples is None else min(len(dataset), max_samples)
    
    for idx in range(num_samples):
        try:
            img, label = dataset[idx]
            if isinstance(img, np.ndarray):
                if img.shape[0] == 3:
                    img = np.transpose(img, (1, 2, 0))
                img = Image.fromarray(img)
            img.save(f'{DATA_ROOT}/{domain}/{split}/{label}/{idx}.png')
            if (idx + 1) % 20000 == 0:
                print(f"{idx + 1}/{num_samples}", end='\r')
        except:
            continue
    print(f"{domain}/{split}: {num_samples} images")

# Check if already exists
already_exists = all(os.path.exists(f'{DATA_ROOT}/{d}/train') for d in domains)

if not already_exists:
    print("Downloading MNIST")
    mnist_train = datasets.MNIST(f'{DATA_ROOT}/mnist_raw', train=True, download=True)
    mnist_test = datasets.MNIST(f'{DATA_ROOT}/mnist_raw', train=False, download=True)
    organize_dataset(mnist_train, 'mnist', 'train')
    organize_dataset(mnist_test, 'mnist', 'test')
    
    print("Downloading SVHN")
    svhn_train = datasets.SVHN(f'{DATA_ROOT}/svhn_raw', split='train', download=True)
    svhn_test = datasets.SVHN(f'{DATA_ROOT}/svhn_raw', split='test', download=True)
    organize_dataset(svhn_train, 'svhn', 'train')
    organize_dataset(svhn_test, 'svhn', 'test')
    
    print("Downloading USPS")
    try:
        from torchvision.datasets import USPS
        usps_train = USPS(f'{DATA_ROOT}/usps_raw', train=True, download=True)
        usps_test = USPS(f'{DATA_ROOT}/usps_raw', train=False, download=True)
        for split, dataset in [('train', usps_train), ('test', usps_test)]:
            for label in range(10):
                os.makedirs(f'{DATA_ROOT}/usps/{split}/{label}', exist_ok=True)
            for idx, (img, label) in enumerate(dataset):
                img.resize((32, 32)).save(f'{DATA_ROOT}/usps/{split}/{label}/{idx}.png')
        print("USPS organized")
    except:
        organize_dataset(mnist_train, 'usps', 'train', max_samples=7291)
        organize_dataset(mnist_test, 'usps', 'test', max_samples=2007)
    
    print("Creating SYN from MNIST")
    for split, mnist_data, max_samples in [('train', mnist_train, 30000), ('test', mnist_test, 5000)]:
        for label in range(10):
            os.makedirs(f'{DATA_ROOT}/syn/{split}/{label}', exist_ok=True)
        for idx in range(max_samples):
            img, label = mnist_data[idx]
            img = img.convert('RGB')
            if random.random() > 0.5:
                img = ImageEnhance.Color(img).enhance(random.uniform(1.2, 2.5))
            img.save(f'{DATA_ROOT}/syn/{split}/{label}/{idx}.png')
            if (idx + 1) % 10000 == 0:
                print(f"    {idx + 1}/{max_samples}", end='\r')
        print(f"syn/{split}")
    
    # Create val folders
    for domain in domains:
        test_path = f'{DATA_ROOT}/{domain}/test'
        val_path = f'{DATA_ROOT}/{domain}/val'
        if os.path.exists(test_path) and not os.path.exists(val_path):
            try:
                os.symlink(test_path, val_path)
            except:
                shutil.copytree(test_path, val_path)
    print("val folders created")
else:
    print("Datasets already exist")

# Verifying Setup

for domain in domains:
    train_path = f'{DATA_ROOT}/{domain}/train'
    val_path = f'{DATA_ROOT}/{domain}/val'
    if os.path.exists(train_path) and os.path.exists(val_path):
        train_count = sum(len(os.listdir(f'{train_path}/{d}')) 
                         for d in os.listdir(train_path) 
                         if os.path.isdir(f'{train_path}/{d}'))
        val_count = sum(len(os.listdir(f'{val_path}/{d}')) 
                       for d in os.listdir(val_path) 
                       if os.path.isdir(f'{val_path}/{d}'))
        print(f"{domain:6s}: train={train_count:6d}, val={val_count:6d}")

print(f"\n GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Not available'}")

In [ ]:
!cd /kaggle/working/RethinkFL && \
python main.py --model fpl --dataset fl_digits --structure simple-cnn \
  --parti_num 10 --communication_epoch 100 --local_epoch 10 \
  --min_participants 5 --max_participants 6 \
  --device_id 0 --seed 42

In [ ]:
!cd /kaggle/working/RethinkFL && \
python main.py --model fedavg --dataset fl_digits --structure simple-cnn \
  --parti_num 10 --communication_epoch 100 --local_epoch 10 \
  --min_participants 5 --max_participants 6 \
  --device_id 0 --seed 42

In [ ]:
!cd /kaggle/working/RethinkFL && \
python main.py --model fedprox --dataset fl_digits --structure simple-cnn \
  --parti_num 10 --communication_epoch 100 --local_epoch 10 \
  --min_participants 5 --max_participants 6 \
  --device_id 0 --seed 42

In [ ]:
!cd /kaggle/working/RethinkFL && \
python main.py --model moon --dataset fl_digits --structure simple-cnn \
  --parti_num 10 --communication_epoch 100 --local_epoch 10 \
  --min_participants 5 --max_participants 6 \
  --device_id 0 --seed 42